# Retrieval-Augmented Generation (RAG)

In this notebook, we explore **Retrieval-Augmented Generation (RAG)**, a powerful technique that combines the strengths of retrieval systems (like BERT embeddings) with generation models (like GPT) to create AI systems that can answer questions using external knowledge.

## Learning Objectives

1. Understand the limitations of pure generation models (hallucination, no external knowledge)
2. Learn the RAG architecture: Retrieve → Augment → Generate
3. Build a complete RAG pipeline with vector databases
4. Measure retrieval quality (Precision@K, Recall@K, MRR, NDCG)
5. Measure generation quality (faithfulness, relevance, correctness)
6. Analyze end-to-end performance (latency, cost, accuracy)

---

## 1. Introduction: Why RAG?

### The Problem with Pure Generation

**GPT-style models** are powerful but have critical limitations:

❌ **Hallucination**: Confidently generates false information  
❌ **No external knowledge**: Only knows training data (with a cutoff date)  
❌ **Cannot cite sources**: No traceability for generated claims  
❌ **Expensive to update**: Retraining costs millions of dollars  

**Example hallucination:**
```
Q: "What is the capital of Atlantis?"
GPT: "The capital of Atlantis is Poseidonia, a magnificent city..."
→ Sounds plausible, but completely made up!
```

### The RAG Solution

RAG solves these problems by combining **retrieval** (finding relevant documents) with **generation** (producing answers).

```
┌─────────────────────────────────────────────────────────────┐
│                      RAG PIPELINE                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. RETRIEVE                                                │
│     Question → Vector DB → Top-K relevant documents         │
│                                                             │
│  2. AUGMENT                                                 │
│     Combine question + retrieved context → Enhanced prompt  │
│                                                             │
│  3. GENERATE                                                │
│     Enhanced prompt → LLM → Answer grounded in context      │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Benefits of RAG

✅ **Grounded answers**: Responses are based on retrieved documents  
✅ **Up-to-date information**: Just add new documents to the database  
✅ **Citable sources**: Can show which documents were used  
✅ **Domain-specific**: Works with private/proprietary data  
✅ **Cost-effective**: No need to retrain large models  

---
## 2. Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets faiss-cpu sentence-transformers torch numpy pandas matplotlib seaborn scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import time
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer, 
    AutoModel,
    GPT2LMHeadModel,
    GPT2Tokenizer,
    set_seed
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import faiss
from sklearn.metrics.pairwise import cosine_similarity

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 3. Building the Knowledge Base

First, we need a collection of documents to retrieve from. We'll use a sample dataset and create a simple knowledge base.

In [ ]:
# Create a sample knowledge base (in production, this would be your documents)
knowledge_base = [
    {
        'id': 1,
        'title': 'Artificial Intelligence',
        'text': 'Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of intelligent agents: any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals.'
    },
    {
        'id': 2,
        'title': 'Machine Learning',
        'text': 'Machine learning (ML) is a field of inquiry devoted to understanding and building methods that learn from data. ML algorithms build a model based on sample data, known as training data, in order to make predictions or decisions without being explicitly programmed to do so.'
    },
    {
        'id': 3,
        'title': 'Deep Learning',
        'text': 'Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning. Deep learning architectures such as deep neural networks, recurrent neural networks, and transformers have been applied to fields including computer vision, speech recognition, and natural language processing.'
    },
    {
        'id': 4,
        'title': 'Natural Language Processing',
        'text': 'Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language. NLP is used to apply machine learning algorithms to text and speech.'
    },
    {
        'id': 5,
        'title': 'Transformers Architecture',
        'text': 'The Transformer is a deep learning architecture that relies on the attention mechanism. It was introduced in the paper Attention Is All You Need in 2017. Transformers have become the foundation for most state-of-the-art NLP models including BERT and GPT.'
    },
    {
        'id': 6,
        'title': 'BERT Model',
        'text': 'BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based machine learning technique for NLP pre-training. It was developed by Google and uses bidirectional attention to understand context from both directions.'
    },
    {
        'id': 7,
        'title': 'GPT Model',
        'text': 'GPT (Generative Pre-trained Transformer) is a family of large language models developed by OpenAI. GPT models use a decoder-only transformer architecture and are trained to predict the next token in a sequence, making them excellent for text generation tasks.'
    },
    {
        'id': 8,
        'title': 'Computer Vision',
        'text': 'Computer vision is an interdisciplinary scientific field that deals with how computers can gain high-level understanding from digital images or videos. It seeks to automate tasks that the human visual system can do, such as object recognition, scene understanding, and image classification.'
    },
    {
        'id': 9,
        'title': 'Reinforcement Learning',
        'text': 'Reinforcement learning (RL) is an area of machine learning concerned with how intelligent agents ought to take actions in an environment to maximize cumulative reward. RL is one of three basic machine learning paradigms, alongside supervised learning and unsupervised learning.'
    },
    {
        'id': 10,
        'title': 'Neural Networks',
        'text': 'A neural network is a series of algorithms that endeavors to recognize underlying relationships in a set of data through a process that mimics the way the human brain operates. Neural networks can adapt to changing input and generate optimal results without requiring redesign of output criteria.'
    }
]

# Convert to DataFrame for easier manipulation
df_kb = pd.DataFrame(knowledge_base)
print(f"Knowledge Base: {len(df_kb)} documents\n")
print(df_kb[['id', 'title']].to_string(index=False))

---
## 4. Step 1: RETRIEVAL - Building the Vector Database

To retrieve relevant documents, we need to:
1. Convert documents to **embeddings** (dense vectors)
2. Store embeddings in a **vector database**
3. Convert queries to embeddings
4. Find most similar documents using **semantic search**

### 4.1 Create Document Embeddings

We'll use **Sentence-BERT** to create high-quality embeddings.

In [ ]:
# Load embedding model (Sentence-BERT)
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and efficient
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding model loaded. Dimension: {embedding_dim}")

In [ ]:
# Generate embeddings for all documents
print("\nGenerating embeddings for documents...")
start_time = time.time()

# Combine title and text for richer embeddings
df_kb['combined_text'] = df_kb['title'] + ': ' + df_kb['text']
documents = df_kb['combined_text'].tolist()

# Encode all documents
doc_embeddings = embedding_model.encode(documents, show_progress_bar=True)
doc_embeddings = np.array(doc_embeddings).astype('float32')

embedding_time = time.time() - start_time
print(f"\nEmbeddings generated in {embedding_time:.2f} seconds")
print(f"Embeddings shape: {doc_embeddings.shape}")

### 4.2 Build FAISS Index

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search in high-dimensional spaces.

In [ ]:
# Create FAISS index
print("Building FAISS index...")
index = faiss.IndexFlatL2(embedding_dim)  # L2 distance (Euclidean)

# Normalize vectors for cosine similarity
faiss.normalize_L2(doc_embeddings)
index.add(doc_embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

### 4.3 Implement Retrieval Function

In [ ]:
def retrieve_documents(query: str, k: int = 3) -> List[Dict]:
    """
    Retrieve top-k most relevant documents for a query.
    
    Args:
        query: The search query
        k: Number of documents to retrieve
    
    Returns:
        List of retrieved documents with scores
    """
    # Encode query
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    faiss.normalize_L2(query_embedding)
    
    # Search in FAISS index
    distances, indices = index.search(query_embedding, k)
    
    # Convert distances to similarity scores (cosine similarity)
    similarities = 1 - distances[0]
    
    # Retrieve documents
    results = []
    for idx, score in zip(indices[0], similarities):
        doc = df_kb.iloc[idx].to_dict()
        doc['retrieval_score'] = float(score)
        results.append(doc)
    
    return results

### 4.4 Test Retrieval

In [ ]:
# Test queries
test_queries = [
    "What is machine learning?",
    "Tell me about transformers in NLP",
    "How does BERT work?",
    "What are neural networks?"
]

print("Testing Retrieval System:\n")
print("="*80)

for query in test_queries:
    print(f"\nQuery: '{query}'\n")
    retrieved = retrieve_documents(query, k=3)
    
    for i, doc in enumerate(retrieved, 1):
        print(f"{i}. {doc['title']} (score: {doc['retrieval_score']:.3f})")
        print(f"   {doc['text'][:100]}...\n")
    print("="*80)

---
## 5. Step 2: AUGMENTATION - Building the Prompt

Now we combine the query with retrieved context to create an augmented prompt.

In [ ]:
def create_augmented_prompt(query: str, retrieved_docs: List[Dict], max_context_length: int = 500) -> str:
    """
    Create an augmented prompt by combining query with retrieved context.
    
    Args:
        query: User's question
        retrieved_docs: List of retrieved documents
        max_context_length: Maximum characters for context
    
    Returns:
        Augmented prompt string
    """
    # Build context from retrieved documents
    context_parts = []
    current_length = 0
    
    for doc in retrieved_docs:
        doc_text = f"[{doc['title']}] {doc['text']}"
        if current_length + len(doc_text) <= max_context_length:
            context_parts.append(doc_text)
            current_length += len(doc_text)
        else:
            break
    
    context = "\n\n".join(context_parts)
    
    # Create prompt template
    prompt = f"""Context:
{context}

Question: {query}

Answer based on the context above:"""
    
    return prompt

In [ ]:
# Example augmented prompt
query = "What is the difference between BERT and GPT?"
retrieved = retrieve_documents(query, k=2)
augmented_prompt = create_augmented_prompt(query, retrieved)

print("Example Augmented Prompt:\n")
print("="*80)
print(augmented_prompt)
print("="*80)

---
## 6. Step 3: GENERATION - Producing the Answer

Now we feed the augmented prompt to a language model to generate an answer.

In [ ]:
# Load generation model (GPT-2)
print("Loading generation model...")
gen_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gen_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
gen_tokenizer.pad_token = gen_tokenizer.eos_token
print("Generation model loaded (GPT-2)")

In [ ]:
def generate_answer(prompt: str, max_length: int = 150) -> str:
    """
    Generate an answer using the language model.
    
    Args:
        prompt: The augmented prompt with context
        max_length: Maximum length of generated answer
    
    Returns:
        Generated answer string
    """
    input_ids = gen_tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # Generate with controlled parameters
    output = gen_model.generate(
        input_ids,
        max_length=input_ids.shape[1] + max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=gen_tokenizer.eos_token_id
    )
    
    # Decode and extract only the new generated part
    full_text = gen_tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract answer (everything after the prompt)
    if "Answer based on the context above:" in full_text:
        answer = full_text.split("Answer based on the context above:")[-1].strip()
    else:
        answer = full_text[len(prompt):].strip()
    
    return answer

---
## 7. Complete RAG Pipeline

In [ ]:
def rag_pipeline(query: str, k: int = 3, verbose: bool = True) -> Dict:
    """
    Complete RAG pipeline: Retrieve → Augment → Generate
    
    Args:
        query: User's question
        k: Number of documents to retrieve
        verbose: Print intermediate steps
    
    Returns:
        Dictionary with answer, retrieved docs, and metadata
    """
    start_time = time.time()
    
    # Step 1: Retrieve
    if verbose:
        print(f"Query: '{query}'\n")
        print("[1/3] Retrieving relevant documents...")
    
    retrieval_start = time.time()
    retrieved_docs = retrieve_documents(query, k=k)
    retrieval_time = time.time() - retrieval_start
    
    if verbose:
        print(f"Retrieved {len(retrieved_docs)} documents in {retrieval_time:.3f}s")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"  {i}. {doc['title']} (score: {doc['retrieval_score']:.3f})")
    
    # Step 2: Augment
    if verbose:
        print("\n[2/3] Creating augmented prompt...")
    
    augment_start = time.time()
    augmented_prompt = create_augmented_prompt(query, retrieved_docs)
    augment_time = time.time() - augment_start
    
    if verbose:
        print(f"Prompt created in {augment_time:.3f}s")
    
    # Step 3: Generate
    if verbose:
        print("\n[3/3] Generating answer...")
    
    generation_start = time.time()
    answer = generate_answer(augmented_prompt)
    generation_time = time.time() - generation_start
    
    total_time = time.time() - start_time
    
    if verbose:
        print(f"Answer generated in {generation_time:.3f}s")
        print(f"\n{'='*80}")
        print(f"ANSWER: {answer}")
        print(f"{'='*80}")
        print(f"\nTotal time: {total_time:.3f}s")
    
    return {
        'query': query,
        'answer': answer,
        'retrieved_docs': retrieved_docs,
        'augmented_prompt': augmented_prompt,
        'retrieval_time': retrieval_time,
        'generation_time': generation_time,
        'total_time': total_time
    }

In [ ]:
# Test the complete RAG pipeline
test_queries_rag = [
    "What is deep learning?",
    "How do transformers work in NLP?",
    "What's the difference between supervised and unsupervised learning?"
]

print("Testing Complete RAG Pipeline:\n")
print("="*80)

for query in test_queries_rag:
    result = rag_pipeline(query, k=2, verbose=True)
    print("\n" + "="*80 + "\n")

---
## 8. Retrieval Metrics

How do we measure if we're retrieving the **right documents**?

### 8.1 Precision@K and Recall@K

**Precision@K**: Of the K retrieved documents, how many are relevant?

$$
\text{Precision@K} = \frac{\text{# relevant docs in top-K}}{K}
$$

**Recall@K**: Of all relevant documents, how many did we retrieve?

$$
\text{Recall@K} = \frac{\text{# relevant docs in top-K}}{\text{total # relevant docs}}
$$

In [ ]:
def calculate_precision_recall_at_k(retrieved_ids: List[int], relevant_ids: List[int], k: int) -> Tuple[float, float]:
    """
    Calculate Precision@K and Recall@K.
    
    Args:
        retrieved_ids: List of retrieved document IDs (top-k)
        relevant_ids: List of ground truth relevant document IDs
        k: Number of retrieved documents
    
    Returns:
        Tuple of (precision@k, recall@k)
    """
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)
    
    relevant_retrieved = retrieved_set.intersection(relevant_set)
    
    precision = len(relevant_retrieved) / k if k > 0 else 0
    recall = len(relevant_retrieved) / len(relevant_set) if len(relevant_set) > 0 else 0
    
    return precision, recall

In [ ]:
# Create evaluation dataset with ground truth
eval_data = [
    {
        'query': 'What is machine learning?',
        'relevant_doc_ids': [2, 1]  # IDs of documents that should be retrieved
    },
    {
        'query': 'Tell me about BERT',
        'relevant_doc_ids': [6, 5]  # BERT and Transformers
    },
    {
        'query': 'How do transformers work?',
        'relevant_doc_ids': [5, 6, 7]  # Transformers, BERT, GPT
    },
    {
        'query': 'What is deep learning?',
        'relevant_doc_ids': [3, 2, 10]  # Deep Learning, ML, Neural Networks
    }
]

# Evaluate retrieval
k_values = [1, 3, 5]
retrieval_metrics = []

for k in k_values:
    precisions = []
    recalls = []
    
    for item in eval_data:
        retrieved = retrieve_documents(item['query'], k=k)
        retrieved_ids = [doc['id'] for doc in retrieved]
        
        prec, rec = calculate_precision_recall_at_k(
            retrieved_ids, 
            item['relevant_doc_ids'], 
            k
        )
        precisions.append(prec)
        recalls.append(rec)
    
    retrieval_metrics.append({
        'K': k,
        'Precision@K': np.mean(precisions),
        'Recall@K': np.mean(recalls)
    })

df_retrieval = pd.DataFrame(retrieval_metrics)
print("\nRetrieval Metrics:\n")
print(df_retrieval.to_string(index=False))

### 8.2 MRR (Mean Reciprocal Rank)

**MRR** measures how quickly we find the first relevant document.

$$
\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}
$$

where $\text{rank}_i$ is the position of the first relevant document for query $i$.

In [ ]:
def calculate_mrr(retrieved_ids: List[int], relevant_ids: List[int]) -> float:
    """
    Calculate Mean Reciprocal Rank.
    
    Args:
        retrieved_ids: List of retrieved document IDs (ordered)
        relevant_ids: List of ground truth relevant document IDs
    
    Returns:
        Reciprocal rank (1/rank of first relevant doc)
    """
    relevant_set = set(relevant_ids)
    
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    
    return 0.0

# Calculate MRR
mrr_scores = []

for item in eval_data:
    retrieved = retrieve_documents(item['query'], k=5)
    retrieved_ids = [doc['id'] for doc in retrieved]
    mrr = calculate_mrr(retrieved_ids, item['relevant_doc_ids'])
    mrr_scores.append(mrr)

avg_mrr = np.mean(mrr_scores)
print(f"\nMean Reciprocal Rank (MRR): {avg_mrr:.3f}")
print("✅ Higher MRR = relevant documents appear earlier in results")

### 8.3 NDCG (Normalized Discounted Cumulative Gain)

**NDCG** accounts for both relevance and position, with diminishing returns for lower ranks.

$$
\text{DCG@K} = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i+1)}
$$

$$
\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}
$$

where $\text{IDCG}$ is the ideal (perfect) DCG.

In [ ]:
from sklearn.metrics import ndcg_score

def calculate_ndcg(retrieved_ids: List[int], relevant_ids: List[int], k: int) -> float:
    """
    Calculate NDCG@K.
    
    Args:
        retrieved_ids: List of retrieved document IDs
        relevant_ids: List of ground truth relevant document IDs
        k: Cutoff position
    
    Returns:
        NDCG@K score
    """
    # Create relevance scores (1 if relevant, 0 otherwise)
    relevance = [1 if doc_id in relevant_ids else 0 for doc_id in retrieved_ids[:k]]
    
    # sklearn expects 2D arrays
    relevance_array = np.array([relevance])
    
    # Create ideal relevance (all relevant docs first)
    ideal_relevance = sorted(relevance, reverse=True)
    ideal_array = np.array([ideal_relevance])
    
    if sum(relevance) == 0:
        return 0.0
    
    return ndcg_score(ideal_array, relevance_array)

# Calculate NDCG@K for different K values
ndcg_results = []

for k in [3, 5, 10]:
    ndcg_scores = []
    
    for item in eval_data:
        retrieved = retrieve_documents(item['query'], k=k)
        retrieved_ids = [doc['id'] for doc in retrieved]
        ndcg = calculate_ndcg(retrieved_ids, item['relevant_doc_ids'], k)
        ndcg_scores.append(ndcg)
    
    ndcg_results.append({
        'K': k,
        'NDCG@K': np.mean(ndcg_scores)
    })

df_ndcg = pd.DataFrame(ndcg_results)
print("\nNDCG Scores:\n")
print(df_ndcg.to_string(index=False))

---
## 9. Generation Quality Metrics

Beyond retrieval, we need to measure if the **generated answer is good**.

### 9.1 Faithfulness (Context Grounding)

Does the answer stay faithful to the retrieved context? (Not hallucinate)

In [ ]:
def calculate_faithfulness(answer: str, context: str) -> float:
    """
    Simple faithfulness check: what fraction of answer tokens appear in context?
    
    In production, use more sophisticated methods like:
    - NLI models (Natural Language Inference)
    - LLM-as-judge
    - BERT-based entailment scoring
    
    Args:
        answer: Generated answer
        context: Retrieved context
    
    Returns:
        Faithfulness score (0-1)
    """
    answer_tokens = set(answer.lower().split())
    context_tokens = set(context.lower().split())
    
    if len(answer_tokens) == 0:
        return 0.0
    
    overlap = answer_tokens.intersection(context_tokens)
    faithfulness = len(overlap) / len(answer_tokens)
    
    return faithfulness

### 9.2 Answer Relevance

Does the answer actually address the question?

In [ ]:
def calculate_answer_relevance(question: str, answer: str) -> float:
    """
    Calculate semantic similarity between question and answer.
    
    Args:
        question: User's question
        answer: Generated answer
    
    Returns:
        Relevance score (0-1)
    """
    # Encode question and answer
    embeddings = embedding_model.encode([question, answer])
    
    # Calculate cosine similarity
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    
    return float(similarity)

### 9.3 Answer Correctness (with ground truth)

If we have reference answers, we can compute similarity scores.

In [ ]:
def calculate_answer_correctness(predicted: str, reference: str) -> Dict[str, float]:
    """
    Calculate multiple correctness metrics.
    
    Args:
        predicted: Generated answer
        reference: Ground truth answer
    
    Returns:
        Dictionary with multiple metrics
    """
    # Semantic similarity
    embeddings = embedding_model.encode([predicted, reference])
    semantic_sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    
    # Token overlap (simple F1)
    pred_tokens = set(predicted.lower().split())
    ref_tokens = set(reference.lower().split())
    
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        token_f1 = 0.0
    else:
        overlap = pred_tokens.intersection(ref_tokens)
        precision = len(overlap) / len(pred_tokens)
        recall = len(overlap) / len(ref_tokens)
        token_f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'semantic_similarity': float(semantic_sim),
        'token_f1': token_f1
    }

---
## 10. End-to-End Evaluation

Let's evaluate the complete RAG pipeline on a test set.

In [ ]:
# Create evaluation dataset with questions and reference answers
eval_qa_data = [
    {
        'query': 'What is machine learning?',
        'reference_answer': 'Machine learning is a field that uses algorithms to learn from data and make predictions without explicit programming.',
        'relevant_doc_ids': [2, 1]
    },
    {
        'query': 'What is BERT?',
        'reference_answer': 'BERT is a bidirectional transformer model developed by Google for NLP tasks that uses masked language modeling.',
        'relevant_doc_ids': [6, 5]
    },
    {
        'query': 'What is deep learning?',
        'reference_answer': 'Deep learning is a subset of machine learning based on artificial neural networks with multiple layers.',
        'relevant_doc_ids': [3, 2, 10]
    }
]

print("Running End-to-End RAG Evaluation...\n")
print("="*80)

evaluation_results = []

for item in eval_qa_data:
    # Run RAG pipeline
    result = rag_pipeline(item['query'], k=3, verbose=False)
    
    # Retrieval metrics
    retrieved_ids = [doc['id'] for doc in result['retrieved_docs']]
    prec, rec = calculate_precision_recall_at_k(retrieved_ids, item['relevant_doc_ids'], 3)
    mrr = calculate_mrr(retrieved_ids, item['relevant_doc_ids'])
    
    # Context for faithfulness
    context = " ".join([doc['text'] for doc in result['retrieved_docs']])
    
    # Generation metrics
    faithfulness = calculate_faithfulness(result['answer'], context)
    relevance = calculate_answer_relevance(item['query'], result['answer'])
    correctness = calculate_answer_correctness(result['answer'], item['reference_answer'])
    
    evaluation_results.append({
        'Query': item['query'][:40] + '...',
        'Precision@3': f"{prec:.2f}",
        'Recall@3': f"{rec:.2f}",
        'MRR': f"{mrr:.2f}",
        'Faithfulness': f"{faithfulness:.2f}",
        'Relevance': f"{relevance:.2f}",
        'Semantic Sim': f"{correctness['semantic_similarity']:.2f}",
        'Token F1': f"{correctness['token_f1']:.2f}",
        'Latency (s)': f"{result['total_time']:.2f}"
    })

df_eval = pd.DataFrame(evaluation_results)
print("\nRAG PIPELINE EVALUATION RESULTS\n")
print(df_eval.to_string(index=False))
print("\n" + "="*80)

### Metrics Summary

In [ ]:
# Calculate average metrics
summary_metrics = {
    'Metric': ['Precision@3', 'Recall@3', 'MRR', 'Faithfulness', 'Relevance', 'Semantic Sim', 'Token F1', 'Avg Latency (s)'],
    'Average Score': [
        df_eval['Precision@3'].astype(float).mean(),
        df_eval['Recall@3'].astype(float).mean(),
        df_eval['MRR'].astype(float).mean(),
        df_eval['Faithfulness'].astype(float).mean(),
        df_eval['Relevance'].astype(float).mean(),
        df_eval['Semantic Sim'].astype(float).mean(),
        df_eval['Token F1'].astype(float).mean(),
        df_eval['Latency (s)'].astype(float).mean()
    ]
}

df_summary = pd.DataFrame(summary_metrics)
df_summary['Average Score'] = df_summary['Average Score'].round(3)

print("\n📊 AVERAGE METRICS ACROSS ALL QUERIES\n")
print(df_summary.to_string(index=False))

In [ ]:
# Visualize metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Retrieval vs Generation Metrics
retrieval_metrics = df_summary[df_summary['Metric'].isin(['Precision@3', 'Recall@3', 'MRR'])]
generation_metrics = df_summary[df_summary['Metric'].isin(['Faithfulness', 'Relevance', 'Semantic Sim'])]

axes[0].barh(retrieval_metrics['Metric'], retrieval_metrics['Average Score'], color='skyblue', label='Retrieval')
axes[0].set_xlabel('Score', fontsize=12)
axes[0].set_title('Retrieval Metrics', fontsize=14, fontweight='bold')
axes[0].set_xlim([0, 1])
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(generation_metrics['Metric'], generation_metrics['Average Score'], color='lightcoral', label='Generation')
axes[1].set_xlabel('Score', fontsize=12)
axes[1].set_title('Generation Quality Metrics', fontsize=14, fontweight='bold')
axes[1].set_xlim([0, 1])
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 11. Performance Analysis: Latency & Cost

Real-world RAG systems must balance **quality**, **latency**, and **cost**.

In [ ]:
# Break down latency components
latency_breakdown = []

for item in eval_qa_data:
    result = rag_pipeline(item['query'], k=3, verbose=False)
    
    latency_breakdown.append({
        'Query': item['query'][:30],
        'Retrieval (s)': result['retrieval_time'],
        'Generation (s)': result['generation_time'],
        'Total (s)': result['total_time']
    })

df_latency = pd.DataFrame(latency_breakdown)

print("\nLatency Breakdown:\n")
print(df_latency.to_string(index=False))

# Calculate percentages
avg_retrieval = df_latency['Retrieval (s)'].mean()
avg_generation = df_latency['Generation (s)'].mean()
total_avg = df_latency['Total (s)'].mean()

print(f"\n📊 Average Latency:")
print(f"  Retrieval: {avg_retrieval:.3f}s ({avg_retrieval/total_avg*100:.1f}%)")
print(f"  Generation: {avg_generation:.3f}s ({avg_generation/total_avg*100:.1f}%)")
print(f"  Total: {total_avg:.3f}s")

In [ ]:
# Visualize latency breakdown
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

x = np.arange(len(df_latency))
width = 0.35

ax.bar(x - width/2, df_latency['Retrieval (s)'], width, label='Retrieval', color='skyblue')
ax.bar(x + width/2, df_latency['Generation (s)'], width, label='Generation', color='lightcoral')

ax.set_xlabel('Query', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('RAG Pipeline Latency Breakdown', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(range(1, len(df_latency)+1))
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Cost Analysis (Approximate)

In production with API-based LLMs (GPT-4, Claude, etc.):

In [ ]:
# Simulate cost calculation (approximate)
cost_comparison = pd.DataFrame([
    {
        'Model': 'GPT-4 (no RAG)',
        'Input Tokens': 100,
        'Output Tokens': 150,
        'Cost per 1M Input': '$5.00',
        'Cost per 1M Output': '$15.00',
        'Cost per Query': '$0.00275'
    },
    {
        'Model': 'GPT-4 (with RAG)',
        'Input Tokens': 500,  # includes retrieved context
        'Output Tokens': 150,
        'Cost per 1M Input': '$5.00',
        'Cost per 1M Output': '$15.00',
        'Cost per Query': '$0.00475'
    },
    {
        'Model': 'GPT-3.5 (with RAG)',
        'Input Tokens': 500,
        'Output Tokens': 150,
        'Cost per 1M Input': '$0.50',
        'Cost per 1M Output': '$1.50',
        'Cost per Query': '$0.00048'
    }
])

print("\n💰 Cost Comparison (Approximate):\n")
print(cost_comparison.to_string(index=False))
print("\n⚠️ RAG increases input tokens but provides grounded, accurate answers")
print("   Trade-off: Higher cost vs lower hallucination")

---
## 12. Practical Insights & Best Practices

### When to Use RAG

✅ **Good for:**
- Q&A over documents/knowledge bases
- Customer support with product documentation
- Research assistants with citation needs
- Domain-specific applications (legal, medical, technical)
- Applications requiring up-to-date information
- Systems that need explainability/traceability

### RAG vs Fine-tuning

| Aspect | RAG | Fine-tuning |
|--------|-----|-------------|
| **Cost** | Low (no training) | High (GPU hours, data prep) |
| **Updates** | Easy (add docs) | Hard (retrain) |
| **Transparency** | High (can cite sources) | Low (black box) |
| **Latency** | Higher (retrieval + generation) | Lower (just generation) |
| **Use Case** | Knowledge-intensive tasks | Task-specific behavior |

### Optimization Strategies

1. **Improve Retrieval:**
   - Better embeddings (domain-specific models)
   - Hybrid search (keyword + semantic)
   - Query expansion/rewriting
   - Re-ranking retrieved documents

2. **Improve Generation:**
   - Better prompts (few-shot examples)
   - Larger/better LLMs
   - Fine-tune generation model
   - Post-processing (fact-checking)

3. **Reduce Latency:**
   - Cache frequent queries
   - Approximate nearest neighbor search (FAISS)
   - Smaller/faster LLMs
   - Parallel processing

4. **Reduce Cost:**
   - Use cheaper models (GPT-3.5 vs GPT-4)
   - Reduce context length
   - Batch queries
   - Self-hosted models

### Common Pitfalls

❌ **Retrieval failures**: Wrong documents retrieved → bad answers  
❌ **Context length limits**: Can't fit all relevant docs  
❌ **Hallucination still possible**: LLM may ignore context  
❌ **Chunking issues**: Breaking documents at wrong boundaries  
❌ **Embedding drift**: Query-document embedding mismatch  

### Production Considerations

1. **Monitoring**: Track retrieval precision, answer quality, latency
2. **A/B Testing**: Test different retrieval/generation strategies
3. **Human-in-the-loop**: Flag low-confidence answers for review
4. **Feedback loops**: Learn from user corrections
5. **Safety**: Content filtering, PII detection

---
## Summary

### Key Takeaways

1. **RAG = Retrieve + Augment + Generate**
   - Solves hallucination and knowledge cutoff problems
   - Combines best of search and generation

2. **Retrieval Metrics:**
   - Precision@K, Recall@K: basic relevance
   - MRR: position of first relevant doc
   - NDCG: accounts for ranking quality

3. **Generation Metrics:**
   - Faithfulness: grounded in context?
   - Relevance: answers the question?
   - Correctness: matches ground truth?

4. **Performance:**
   - Retrieval is fast (<0.1s)
   - Generation dominates latency
   - Cost scales with context + output length

5. **Production:**
   - Monitor quality continuously
   - Balance cost, latency, accuracy
   - Provide citations/sources

### Next Steps

**In NLP14_2_Modern_LLMs.ipynb**, we'll explore:
- Using production LLM APIs (GPT-4, Claude)
- Advanced prompt engineering
- Few-shot learning
- LLM evaluation frameworks

---

### Further Reading

- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (Lewis et al., 2020)](https://arxiv.org/abs/2005.11401)
- [LangChain RAG Documentation](https://python.langchain.com/docs/use_cases/question_answering/)
- [FAISS: The Missing Manual](https://github.com/facebookresearch/faiss/wiki)
- [Pinecone Learning Center: Vector Databases](https://www.pinecone.io/learn/)